# 🔌 Pipeline: Comunicación Serial Python ↔ Arduino
**End-to-end: Serial Communication between Python and Arduino**  
Diplomado Superior en Redes Neuronales Artificiales y Deep Learning

---
> Este pipeline cubre el flujo completo de comunicación serial entre Python y Arduino:  
> detección de puertos, envío/recepción de datos, simulación sin hardware, y logging.

---
## 📡 Introducción a la Comunicación Serial

La comunicación serial permite intercambiar datos entre Python y Arduino a través del puerto USB.  
**Conceptos clave:**
- **TX/RX**: Transmisión y Recepción
- **Baudrate**: Velocidad de comunicación (9600 bps es típico)
- **ASCII**: Los datos se envían como texto codificado
- **Protocolo**: Las reglas que acordamos para interpretar los mensajes

```
Python ──→ Serial.write() ──→ Arduino (Serial.read())
Arduino ──→ Serial.println() ──→ Python (readline())
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..')))
from libreria_modulo5 import *

---
## 🔍 Detectar Puertos Disponibles

Escanea los puertos COM del sistema. Requiere tener instalado `pyserial`.

In [ ]:
puertos = detectar_puertos()
print(f"Puertos encontrados: {puertos}")

---
## 🤖 Simulación con ArduinoSimulado

Sin hardware físico, podemos simular un Arduino para practicar el protocolo serial.

In [ ]:
# Crear y configurar el Arduino simulado
arduino = ArduinoSimulado("MiArduino")
arduino.configurar(9600)

In [ ]:
# Enviar comandos con serial_println
arduino.serial_println("Sistema iniciado")
arduino.serial_println("Sensor listo")
arduino.serial_println("Valor: 42")

In [ ]:
# Leer respuestas del buffer serial
print("Leyendo buffer serial:")
while arduino.serial_available() > 0:
    msg = arduino.serial_read()
    print(f"  → Recibido: {msg}")

In [ ]:
# Simular un ciclo completo: preguntar y recibir
print("=== Ciclo comando-respuesta ===")
comandos = ["LED_ON", "LED_OFF", "STATUS", "TEMPERATURA"]
for cmd in comandos:
    print(f"  >> Enviando: {cmd}")
    arduino.serial_println(f"Recibido: {cmd.upper()}")
    respuesta = arduino.serial_read()
    print(f"  << {respuesta}")

---
## 🔌 Conexión Real con Arduino (Requiere Hardware)

El siguiente código está comentado porque requiere un Arduino físicamente conectado.  
Descomenta y ajusta el puerto COM según tu sistema.

In [ ]:
# Descomenta para usar con Arduino real:
# ser = conectar_arduino('COM3', 9600)
# if ser:
#     enviar(ser, 'LED_ON')
#     respuesta = leer_linea(ser)
#     print(f"Respuesta: {respuesta}")
#     cerrar(ser)

print("☑️  Código comentado — conecta tu Arduino y descomenta para probar")

---
## 📊 Serial Data Logger — Simulación

Simula la lectura de 10 valores analógicos desde el Arduino (como si fueran sensores).

In [ ]:
import time
import random

print("=== Serial Data Logger (Simulado) ===")
print("Timestamp\t\t| Sensor_A\t| Sensor_B\t| Sensor_C")
print("-" * 60)

for i in range(10):
    ts = time.strftime("%H:%M:%S")
    sA = random.randint(100, 900)
    sB = random.randint(200, 800)
    sC = random.randint(0, 1023)
    arduino.serial_println(f"{ts},{sA},{sB},{sC}")
    print(f"{ts}\t\t| {sA}\t\t| {sB}\t\t| {sC}")
    time.sleep(0.2)

print("\n✅ 10 lecturas registradas en el buffer serial")

In [ ]:
# Leer todas las líneas del buffer
print("=== Leyendo datos del buffer ===")
while arduino.serial_available() > 0:
    linea = arduino.serial_read()
    print(f"  📥 {linea}")

---
## 🔧 Sketch Arduino (C++) para el Receptor

Este código se carga en el Arduino para recibir comandos desde Python.

```cpp
// Arduino Receptor Serial
// Copia este código en Arduino IDE y cárgalo en tu placa

void setup() {
    Serial.begin(9600);
    pinMode(LED_BUILTIN, OUTPUT);
    Serial.println("Arduino listo");
}

void loop() {
    if (Serial.available() > 0) {
        String comando = Serial.readStringUntil('\n');
        comando.trim();

        if (comando == "LED_ON") {
            digitalWrite(LED_BUILTIN, HIGH);
            Serial.println("LED encendido");
        }
        else if (comando == "LED_OFF") {
            digitalWrite(LED_BUILTIN, LOW);
            Serial.println("LED apagado");
        }
        else if (comando == "STATUS") {
            Serial.println("OK");
        }
        else if (comando == "TEMPERATURA") {
            int valor = analogRead(A0);
            Serial.println(valor);
        }
        else {
            Serial.print("Comando no reconocido: ");
            Serial.println(comando);
        }
    }
}
```

---
## 📦 Protocolo Binario (Concepto)

Cuando envías texto, cada número ocupa varios bytes. Para estructuras de datos grandes,  
conviene usar **protocolo binario**: empaquetar los datos en bytes con formato fijo.

### Ejemplo: envío estructurado con struct
```python
import struct
# Empaquetar: entero 16 bits + entero 16 bits + float
datos = struct.pack('HHf', sensor1, sensor2, temperatura)
ser.write(datos)  # Solo 8 bytes en vez de texto
```

### Ventajas del protocolo binario
- **Eficiencia**: menos bytes transmitidos
- **Determinismo**: estructura conocida, sin parseo
- **Velocidad**: ideal para alta frecuencia de muestreo

### En Arduino se recibe así:
```cpp
union {
    byte bytes[8];
    struct {
        uint16_t sensor1;
        uint16_t sensor2;
        float temperatura;
    } datos;
} paquete;

if (Serial.available() >= 8) {
    Serial.readBytes(paquete.bytes, 8);
    // Usar: paquete.datos.sensor1, paquete.datos.temperatura
}
```

---
## 🧪 #TODO: Ejercicios

1. **Protocolo propio**: Define un protocolo simple con comandos `SET_PIN 13 1` y `GET_ANALOG A0`.  
   Implementa el parseo en el sketch Arduino.

2. **Data logger real**: Conecta un sensor real (LM35, DHT11, potenciómetro) y registra 100 lecturas.  
   Calcula promedio, mínimo y máximo desde Python.

3. **Serial plotter**: Usa `matplotlib` para graficar en tiempo real los valores que llegan por serial.

4. **Handshake**: Implementa un protocolo de sincronización:  
   Python envía `SYNC`, Arduino responde `ACK`, y comienza la transmisión.

5. **Binario real**: Envía 3 sensores como `struct.pack('HHH', ...)` y recíbelos en Arduino.  
   Mide cuántos bytes ahorras vs. texto.